In [8]:
import sys
import os
from pathlib import Path

from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath(os.path.join('/home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering')))
#Import your class
from src.features.build_features import FeatureEngineer
import pandas as pd

#Load your dataset
df = pd.read_csv('/home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering/raw data/rt_data.csv', parse_dates=['InvoiceDate'])

#Initialize the feature engineer
fe = FeatureEngineer()

#Create RFM features
rfm_df = fe.create_rfm_features(df)

#Inspect the output
rfm_df.head(20)
rfm_df.describe()

# 7. Save feature engineer (for reuse in pipeline / deployment)
fe.save_engineer("/home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering/models/feature_engineer.pkl")

Engineering RFM features...
Feature Engineering complete. Shape: (3256, 5)
FeatureEngineer saved successfully to /home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering/models/feature_engineer.pkl


In [12]:
from src.Transformer.Preprocessing import DataPreprocessor
from sklearn.preprocessing import StandardScaler

# Initialize preprocessor
preprocessor = DataPreprocessor(test_ratio=0.2, scaler=StandardScaler())

# Run full preprocessing pipeline
train_df, test_df = preprocessor.run_pipeline(rfm_df)

#Inspect results
print("Training set:")
print(train_df.head())
print("\nTest set:")
print(test_df.head())

# Save preprocessor for reuse in deployment
preprocessor.save_preprocessor("/home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering/models/preprocessor.pkl")


Preprocessing Complete. Training shape: (2605, 5)
Training set:
             Recency    Tenure  Frequency  Monetary  AvgOrderValue
CustomerID                                                        
13933.0    -1.288004 -1.756014   0.061462  0.564046       0.601462
17703.0    -1.288004 -2.075268   0.061462  0.415162       0.405756
12363.0    -1.288004 -0.093221   0.061462  0.402951       0.389708
16555.0    -1.288004 -3.574434  -0.860545 -0.663955      -0.174550
16013.0    -1.288004 -2.906333   0.061462 -0.107828      -0.281120

Test set:
             Recency    Tenure  Frequency  Monetary  AvgOrderValue
CustomerID                                                        
18215.0    -1.554849 -7.850022  -0.860545 -0.148260       0.504597
17377.0    -1.554849  0.999022   3.015963  1.645190       0.070931
14432.0    -1.554849 -7.850022  -0.860545 -0.251202       0.369026
14508.0    -1.554849 -7.850022  -0.860545 -1.062640      -0.699601
14533.0    -1.554849  0.863042   0.715636  0.547743   

In [14]:
from src.models.train_pipeline import KMeansTrainer

# Assume train_df and test_df are preprocessed already
trainer = KMeansTrainer(train_df=train_df, test_df=test_df, k_range=range(3, 9))
trainer.fit()

# Assign segments
train_segmented, test_segmented = trainer.assign_segments()

# Save best model
trainer.save_model("/home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering/models/kmeans_best.pkl")

# Access metrics
print(trainer.metrics)

k=3: Train Silhouette=0.3465, Test Silhouette=0.5528
k=4: Train Silhouette=0.3371, Test Silhouette=0.4468
k=5: Train Silhouette=0.2928, Test Silhouette=0.3062
k=6: Train Silhouette=0.2904, Test Silhouette=0.2778
k=7: Train Silhouette=0.2837, Test Silhouette=0.2057
k=8: Train Silhouette=0.2731, Test Silhouette=0.2763
Best K=3 with Train Silhouette=0.3465
Best KMeans model saved at /home/selowa-mphadi/PycharmProjects/pythonProject/kmeans clustering/models/kmeans_best.pkl
{3: {'train_silhouette': 0.34645860411861784, 'test_silhouette': 0.5527914481295587}, 4: {'train_silhouette': 0.3371436706852542, 'test_silhouette': 0.44683357881602653}, 5: {'train_silhouette': 0.2927727779271959, 'test_silhouette': 0.3062460991887189}, 6: {'train_silhouette': 0.29040064142759103, 'test_silhouette': 0.27777477190911387}, 7: {'train_silhouette': 0.2836669738909367, 'test_silhouette': 0.2057056662917828}, 8: {'train_silhouette': 0.2730652327703699, 'test_silhouette': 0.2763159034375256}}


In [15]:
train_df

,Recency,Tenure,Frequency,Monetary,AvgOrderValue
CustomerID,,,,,
13933.0,-1.288004,-1.756014,0.061462,0.564046,0.601462
17703.0,-1.288004,-2.075268,0.061462,0.415162,0.405756
12363.0,-1.288004,-0.093221,0.061462,0.402951,0.389708
16555.0,-1.288004,-3.574434,-0.860545,-0.663955,-0.174550
16013.0,-1.288004,-2.906333,0.061462,-0.107828,-0.281120
...,...,...,...,...,...
13255.0,2.084914,0.999022,-0.860545,-1.574392,-1.373554
14237.0,2.084914,0.999022,-0.860545,-1.280930,-0.987079
17643.0,2.084914,0.999022,-0.860545,-1.531709,-1.317344


In [16]:
print(preprocessor.scaler.feature_names_in_)

['Recency' 'Tenure' 'Frequency' 'Monetary' 'AvgOrderValue']
